# inspecionar conteúdo do treino

In [ ]:
import numpy as np
import os
import glob
import re

# ==============================================================================
# CONFIGURAÇÃO DE LEITURA
# ==============================================================================

# Defina o diretório base da SEQUÊNCIA.
# ESTE CAMINHO DEVE TERMINAR na pasta seqXXXX (ex: .../train/seq0001).
DATA_ROOT = r'C:\Users\Micro\Documents\sourcecode\MMNTT\train\seq0103'

# As variáveis abaixo são usadas apenas para impressão, não para construção de caminho,
# pois DATA_ROOT já define a localização.
DEFAULT_SPLIT = 'train' 
DEFAULT_SEQ_NUMBER = '0100'

# ==============================================================================
# FUNÇÕES PRINCIPAIS
# ==============================================================================

def read_ground_truth_class_fixed(root_dir: str, split_name: str, seq_number_str: str):
    """
    Lê o Ground Truth (GT) da classe para a sequência especificada em root_dir e imprime.

    Args:
        root_dir: Caminho completo para a pasta seqXXXX (ex: .../seq0001).
        split_name: Nome do split (ex: 'train') para impressão.
        seq_number_str: Número da sequência (ex: '0001') para impressão.
    """
    # O caminho correto é apenas root_dir + 'class'
    class_dir = os.path.join(root_dir, 'class')

    print("=" * 70)
    print(f"Lendo Ground Truth de Classe (GT) para: {split_name}/seq{seq_number_str}")
    print(f"Buscando em: {class_dir}")
    print("=" * 70)
    
    # 1. Verificação de Caminho
    if not os.path.exists(class_dir):
        print(f"ERRO: Diretório de classes não encontrado em: {class_dir}")
        print("Verifique se 'DATA_ROOT' está apontando para a pasta correta da sequência.")
        return

    # 2. Busca e Carregamento de Arquivos
    # Busca por todos os arquivos .npy no diretório 'class'
    class_files = glob.glob(os.path.join(class_dir, "*.npy"))
    
    if not class_files:
        print(f"AVISO: Nenhuma informação de Ground Truth de classe (.npy) encontrada em {class_dir}.")
        return

    results = []
    
    for file_path in class_files:
        filename = os.path.basename(file_path)
        
        # O timestamp é o nome do arquivo (excluindo a extensão .npy)
        timestamp_str = os.path.splitext(filename)[0]
        
        try:
            timestamp = float(timestamp_str)
            # O GT da classe é carregado como um array numpy e .item() extrai o inteiro
            class_gt = np.load(file_path).item()
            
            # Mapeamento simples para legibilidade
            if 0 <= class_gt <= 3:
                class_label = f"Drone Tipo {class_gt}"
            else:
                class_label = f"Classe Desconhecida/Fundo ({class_gt})"

            results.append({
                'timestamp': timestamp,
                'class_gt': class_gt,
                'class_label': class_label
            })
            
        except Exception as e:
            print(f"Erro ao ler o arquivo {filename}: {e}")

    # 3. Ordenação e Exibição
    results.sort(key=lambda x: x['timestamp'])

    print(f"| {'Timestamp':<20} | {'GT Classe (Int)':<15} | {'Drone/Tipo':<25} |")
    print("-" * 70)
    for res in results:
        # Imprime os resultados no formato solicitado
        print(f"| {res['timestamp']:.6f} | {res['class_gt']:<15} | {res['class_label']:<25} |")

    print("=" * 70)
    print(f"Total de {len(results)} frames de classes lidos.")


# ==============================================================================
# PONTO DE ENTRADA
# ==============================================================================

if __name__ == '__main__':
    # Executa a função usando a configuração definida em DATA_ROOT
    read_ground_truth_class_fixed(DATA_ROOT, DEFAULT_SPLIT, DEFAULT_SEQ_NUMBER)


# inspecionar dataloader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
from PIL import Image
from collections import defaultdict
import math
from typing import List, Tuple, Dict, Any, Optional
import random
from sklearn.model_selection import train_test_split
from tabulate import tabulate # Para uma saída formatada e clara

# =========================================================================
# 🎯 VARIÁVEIS DE CONFIGURAÇÃO DO INSPECTOR
# =========================================================================

# ⚠️ DEFINA O CAMINHO RAIZ:
# Substitua o valor abaixo pelo caminho ABSOLUTO da sua pasta 'Mavic3'.
DATA_ROOT = r"C:\Users\Micro\Documents\sourcecode\MMNTT\Mavic3" 

SAMPLES_TO_INSPECT = 2 # Quantidade de amostras para inspecionar em detalhes (use um valor pequeno)

# Configuração Padrão do Dataset (deve ser coerente com o treino)
CONFIG = {
    'image_size': 224,
    'modal_feature_dim': 512, # A dimensão final das features de PCL (Lidar, Livox, Radar)
    'sequence_length': 1,     # 1 para inspeção (não chunks BPTT)
}

# Configuração de Log: Mude para True para ver as mensagens de sincronização/fallback
LOG_INCONSISTENCY = False 

# Lista das subpastas/modalidades fixas dentro de 'Mavic3'
MODALITIES = [
    'ground_truth',
    'image',
    'lidar_360',
    'livox_avia',
    'radar_enhance_pcl'
]

# Mapeamento do nome da pasta para a chave no dicionário de saída
OUTPUT_KEY_MAP = {
    'ground_truth': 'gt_pos',
    'image': 'image',
    'lidar_360': 'lidar_360',
    'livox_avia': 'livox_avia',
    'radar_enhance_pcl': 'radar'
}
DEFAULT_FEATURE_DIM = CONFIG['modal_feature_dim']

# =========================================================================
# ⚙️ FUNÇÕES DE CARREGAMENTO DE DADOS (Mavic3Dataset e get_mavic3_datasets)
# =========================================================================

class Mavic3Dataset(Dataset):
    """
    Custom PyTorch Dataset para carregar dados de múltiplas modalidades 
    da pasta Mavic3, sincronizando amostras por timestamp.
    """
    def __init__(self, data_root: str, all_timestamps: List[str], config: Dict[str, Any]):
        self.data_root = data_root
        self.config = config
        self.img_size = config.get('image_size', 224)
        self.feature_dim = config.get('modal_feature_dim', DEFAULT_FEATURE_DIM)
        self.sequence_length = config.get('sequence_length', 1) 
        self.all_timestamps = all_timestamps 

        self.modality_paths = {
            mod: os.path.join(self.data_root, mod) for mod in MODALITIES
        }
        
        for mod, path in self.modality_paths.items():
            if not os.path.isdir(path):
                # Este erro garante que todas as pastas de modalidade existam
                raise FileNotFoundError(f"Subpasta de modalidade '{mod}' não encontrada em: {self.data_root}")

        print(f"Dataset inicializado com {len(self.all_timestamps)} amostras.")
        
    def __len__(self):
        if self.sequence_length > 1:
            return math.floor(len(self.all_timestamps) / self.sequence_length)
        else:
            return len(self.all_timestamps)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        
        # Lógica para carregar a(s) amostra(s)
        if self.sequence_length > 1:
            start_index = index * self.sequence_length
            end_index = start_index + self.sequence_length
            chunk_timestamps = self.all_timestamps[start_index:end_index]
        else:
            chunk_timestamps = [self.all_timestamps[index]]

        sequence_data = defaultdict(list)
        
        for timestamp_id in chunk_timestamps:
            sample_data = self._load_single_sample(timestamp_id)
            for key, value in sample_data.items():
                sequence_data[key].append(value)
        
        stacked_data = {
            # Se sequence_length=1, o stack adiciona a dimensão T=1, resultando em (T, C, H, W) ou (T, D)
            key: torch.stack(value, dim=0) 
            for key, value in sequence_data.items()
            if key != 'timestamp' 
        }
        
        stacked_data['timestamp'] = sequence_data.get('timestamp', chunk_timestamps)
        
        # DERIVAÇÃO DO GT DE CLASSE A PARTIR DO GT DE POSIÇÃO
        gt_pos_tensor = stacked_data['gt_pos']
        # Calcula a distância da posição ([T, 3]) ao ponto [0, 0, 0]
        # Nota: O torch.linalg.norm é mantido, mas o GT deve ser indexado corretamente: [:, :]
        if gt_pos_tensor.ndim > 1:
             distance_norm = torch.linalg.norm(gt_pos_tensor, dim=1)
        else:
             distance_norm = torch.linalg.norm(gt_pos_tensor)
        
        # Classe 1 se distância > 1e-6 (Presente), Classe 0 caso contrário (Ausente)
        gt_class_from_pos = (distance_norm > 1e-6).long() 
        stacked_data['gt_class'] = gt_class_from_pos 
        
        return stacked_data

    def _load_single_sample(self, timestamp_id: str) -> Dict[str, Any]:
        sample_data = {}
        sample_data['timestamp'] = timestamp_id
        
        for modality, path in self.modality_paths.items():
            output_key = OUTPUT_KEY_MAP.get(modality, modality)
            
            if modality == 'image':
                # Imagem agora usa a lógica de sincronização _find_closest_file
                sample_data[output_key] = self._load_image(path, timestamp_id)
            elif modality == 'ground_truth':
                gt_pos = self._load_npy(path, timestamp_id, feature_dim=None, gt_dim=3, is_class=False)
                sample_data[output_key] = gt_pos
            elif modality in ['lidar_360', 'livox_avia', 'radar_enhance_pcl']:
                # PCLs (Point Clouds) são processados para features
                data_tensor = self._load_npy(path, timestamp_id, feature_dim=self.feature_dim, gt_dim=None, is_class=False)
                sample_data[output_key] = data_tensor
        
        return sample_data

    def _get_modality_name_from_path(self, base_path: str) -> str:
        return os.path.basename(base_path)

    # 🔄 REFACTOR: Função de sincronização unificada para .npy e .png
    def _find_closest_file(self, base_path: str, timestamp_id: str, extensions: List[str]) -> str:
        """
        Função Crítica de Sincronização: Encontra o arquivo cujo timestamp
        é o mais próximo do timestamp_id de referência, filtrando por extensões.
        """
        data_type = self._get_modality_name_from_path(base_path)
        
        if not os.path.isdir(base_path):
            raise FileNotFoundError(f"Diretório base não encontrado: {base_path}")
            
        try:
            # Filtro pela parte inteira do timestamp para otimização em diretórios grandes.
            timestamp_prefix = timestamp_id.split('.')[0]
        except:
            timestamp_prefix = timestamp_id
            
        candidate_files = []
        for filename in os.listdir(base_path):
            is_match = any(filename.endswith(ext) for ext in extensions)
            if filename.startswith(timestamp_prefix) and is_match:
                try:
                    # Tenta extrair o timestamp do nome do arquivo
                    file_ts_str = filename.rsplit('.', 1)[0]
                    file_ts = float(file_ts_str)
                    candidate_files.append((filename, file_ts))
                except ValueError:
                    continue
                        
        if not candidate_files:
            # Este é o ponto onde a sincronização FALHA
            ext_str = ", ".join(extensions)
            raise FileNotFoundError(f"Nenhum arquivo ({ext_str}) com o prefixo '{timestamp_prefix}' encontrado em {base_path} ({data_type}).")

        reference_ts = float(timestamp_id)
        best_match_file = candidate_files[0][0]
        min_diff = abs(reference_ts - candidate_files[0][1])

        for filename, file_ts in candidate_files[1:]:
            diff = abs(reference_ts - file_ts)
            if diff < min_diff:
                min_diff = diff
                best_match_file = filename
                
        matched_path = os.path.join(base_path, best_match_file)
        
        if LOG_INCONSISTENCY and min_diff > 0.005: # Sinaliza diferença de sincronização > 5ms
            print(f"DEBUG SYNC: {data_type} (Ref: {timestamp_id}) -> Match: {best_match_file} (Diff: {min_diff:.6f}s) - INCONSISTÊNCIA DETECTADA")
            
        return matched_path

    def _load_image(self, base_path: str, timestamp_id: str) -> torch.Tensor:
        """Carrega e pré-processa a imagem (.png) usando sincronização robusta."""
        try:
            # 💡 Ajuste: Usa o find_closest_file para sincronização
            img_path = self._find_closest_file(base_path, timestamp_id, ['.png'])
        except FileNotFoundError as e:
            if LOG_INCONSISTENCY: print(f"AVISO: Falha de sincronização de Imagem em {base_path}. {e}")
            # Retorna tensor de zeros (fallback)
            return torch.zeros((3, self.img_size, self.img_size), dtype=torch.float)

        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((self.img_size, self.img_size))
            return torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        except Exception as e:
            if LOG_INCONSISTENCY: print(f"AVISO: Falha ao carregar/processar imagem {img_path}. {e}")
            # Retorna tensor de zeros (fallback)
            return torch.zeros((3, self.img_size, self.img_size), dtype=torch.float)

    def _load_npy(self, base_path: str, timestamp_id: str, feature_dim: Optional[int] = None, gt_dim: Optional[int] = None, is_class: bool = False) -> torch.Tensor:
        """Carrega e pré-processa os dados .npy (PCL ou Ground Truth)."""
        try:
            # 💡 Ajuste: Usa o find_closest_file
            npy_path = self._find_closest_file(base_path, timestamp_id, ['.npy'])
        except FileNotFoundError as e:
            # Caso de arquivo de modalidade ausente/dessincronizado (FileNotFound dentro de _find_closest)
            if LOG_INCONSISTENCY: print(f"AVISO: Falha de sincronização de NPY em {base_path}. {e}")
            # Retorna o tensor de zeros (fallback)
            if is_class: return torch.tensor(-1, dtype=torch.long)
            if gt_dim: return torch.zeros(gt_dim, dtype=torch.float)
            if feature_dim: return torch.zeros(feature_dim, dtype=torch.float)
            return torch.zeros(1, dtype=torch.float)

        try:
            data = np.load(npy_path, allow_pickle=True)
            if is_class: return torch.tensor(data.item(), dtype=torch.long)
            
            data_tensor = torch.from_numpy(data).float()
            
            # 1. Flatten se for multidimensional (ex: PCLs)
            if data_tensor.ndim > 1:
                data_tensor = data_tensor.flatten()
            
            # 2. Padding/Truncamento para Features (PCLs)
            if feature_dim:
                current_length = data_tensor.size(0)
                if current_length > feature_dim:
                    data_tensor = data_tensor[:feature_dim]
                elif current_length < feature_dim:
                    padding_needed = feature_dim - current_length
                    padding = torch.zeros(padding_needed, dtype=torch.float)
                    data_tensor = torch.cat((data_tensor, padding), 0)
                return data_tensor.reshape(feature_dim)
            
            # 3. Tratamento para GT de Posição (dim 3)
            if gt_dim:
                # Garante que o GT tenha dimensão 3
                if data_tensor.size(0) >= gt_dim:
                    return data_tensor[:gt_dim].float()
                else:
                    padding_needed = gt_dim - data_tensor.size(0)
                    padding = torch.zeros(padding_needed, dtype=torch.float)
                    return torch.cat((data_tensor, padding), 0).float()
                    
            return data_tensor.float()

        except Exception as e:
            if LOG_INCONSISTENCY: print(f"AVISO: Falha ao carregar NPY (Corrupção?) {npy_path}. {e}")
            # Retorna o tensor de zeros (fallback)
            if is_class: return torch.tensor(-1, dtype=torch.long) 
            if gt_dim: return torch.zeros(gt_dim, dtype=torch.float) 
            if feature_dim: return torch.zeros(feature_dim, dtype=torch.float)
            return torch.zeros(1, dtype=torch.float)


def get_mavic3_datasets(data_root: str, config: Dict[str, Any], test_size: float = 0.1, val_size: float = 0.1, random_state: int = 42) -> Dict[str, Mavic3Dataset]:
    """Função auxiliar para carregar todos os timestamps e dividir em Treino, Validação e Teste."""
    
    gt_path = os.path.join(data_root, 'ground_truth')
    if not os.path.isdir(gt_path):
        raise FileNotFoundError(f"Diretório de Ground Truth não encontrado em {gt_path}. Verifique se 'data_root' aponta para a pasta 'Mavic3'.")

    # 1. Carrega todos os timestamps dos arquivos Ground Truth (.npy)
    all_timestamps = sorted([f.rsplit('.', 1)[0] for f in os.listdir(gt_path) if f.endswith('.npy')])
    
    if not all_timestamps:
        raise ValueError(f"Nenhum arquivo .npy encontrado em {gt_path}. O dataset está vazio.")

    # 2. Divisão em Treino, Validação e Teste (usando timestamps como referência)
    train_val_timestamps, test_timestamps = train_test_split(
        all_timestamps, 
        test_size=test_size, 
        random_state=random_state,
        shuffle=True
    )

    val_ratio_in_train_val = val_size / (1 - test_size)
    
    train_timestamps, val_timestamps = train_test_split(
        train_val_timestamps, 
        test_size=val_ratio_in_train_val, 
        random_state=random_state,
        shuffle=True
    )
    
    print("\n" + "="*50)
    print(f"RESUMO GERAL DO DATASET CARREGADO ({len(all_timestamps)} amostras totais)")
    print(f"Caminho Raiz: {data_root}")
    print("="*50)
    print("Divisão do Dataset (GT de Posição):")
    print(f" - Treino: {len(train_timestamps)} amostras")
    print(f" - Validação: {len(val_timestamps)} amostras")
    print(f" - Teste: {len(test_timestamps)} amostras")
    print("-" * 50)

    # 3. Cria as instâncias dos Datasets
    train_dataset = Mavic3Dataset(data_root, train_timestamps, config)
    val_dataset = Mavic3Dataset(data_root, val_timestamps, config)
    test_dataset = Mavic3Dataset(data_root, test_timestamps, config)
    
    return {
        'train': train_dataset,
        'val': val_dataset,
        'test': test_dataset
    }

# =========================================================================
# 🔎 FUNÇÕES DE INSPEÇÃO
# =========================================================================

def inspect_data_structure(data_root: str):
    """Verifica a estrutura de diretórios e a contagem de arquivos por modalidade."""
    print("\n" + "="*50)
    print("1. AUDITORIA DA ESTRUTURA DE DADOS (File System)")
    print("="*50)
    
    data = []
    
    for mod in MODALITIES:
        mod_path = os.path.join(data_root, mod)
        key_name = OUTPUT_KEY_MAP.get(mod, mod)
        
        exists = os.path.isdir(mod_path)
        file_count = 0
        file_extensions = set()
        
        if exists:
            for f in os.listdir(mod_path):
                file_count += 1
                file_extensions.add(os.path.splitext(f)[1])
                
        data.append([
            key_name,
            mod,
            "✅ Sim" if exists else "❌ Não",
            file_count,
            ", ".join(sorted(list(file_extensions)))
        ])

    print(tabulate(data, headers=["Chave de Saída", "Pasta Raiz", "Existe?", "N° Arquivos", "Extensões"], tablefmt="fancy_grid"))
    print("\n⚠️  Observação: O número total de arquivos listado deve ser consistente entre as modalidades, exceto se houver dessincronização esperada.")

def inspect_sample_data(dataloader: DataLoader, dataset_name: str, num_samples: int):
    """Itera sobre as primeiras amostras do DataLoader e inspeciona os tensores."""
    print("\n" + "="*50)
    print(f"2. INSPEÇÃO DETALHADA DAS AMOSTRAS ({dataset_name.upper()})")
    print("="*50)

    for i, batch in enumerate(dataloader):
        if i >= num_samples:
            break
            
        # O DataLoader retorna um batch de tamanho B, onde cada elemento é uma sequência de tamanho T
        # Ex: Imagem -> (B, T, C, H, W)
        print(f"\n--- AMOSTRA N° {i+1}/{num_samples} ---")
        
        # 1. Metadados de Tempo
        # Se T=1, batch.get('timestamp') é uma lista de listas: [['ts1'], ['ts2'], ...]
        # Se B=1, é apenas [['ts1']]
        timestamp = batch.get('timestamp', [['N/A']])[0][0]
        print(f"  > Timestamp Original (GT): {timestamp}")
        print(f"  > Chaves Retornadas: {list(batch.keys())}")
        
        data_table = []
        inconsistency_found = False
        
        # Itera sobre todas as chaves (exceto o timestamp)
        with torch.no_grad():
            for key, tensor in batch.items():
                if key == 'timestamp':
                    continue
                
                # Para inspeção, pegamos o primeiro elemento do Batch (índice 0) 
                # e o primeiro elemento da Sequência (índice 0).
                # Transformando (B, T, ...) -> (...)
                if tensor.ndim > 0:
                    tensor = tensor[0, 0, ...]
                
                shape = list(tensor.shape)
                dtype = str(tensor.dtype).split('.')[-1]
                
                # --- Diagnóstico Rápido ---
                stats_str = ""
                nan_inf_zero = []
                
                if tensor.is_floating_point():
                    np_array = tensor.cpu().numpy()
                    
                    # Check NaNs/Infs
                    if np.isnan(np_array).any():
                        nan_inf_zero.append("NaN")
                        inconsistency_found = True
                    if np.isinf(np_array).any():
                        nan_inf_zero.append("Inf")
                        inconsistency_found = True
                        
                    # Estatísticas
                    mean_val = np_array.mean()
                    std_val = np_array.std()
                    min_val = np_array.min()
                    max_val = np_array.max()

                    # Zeroed Check (Verifica se todos os valores são ~zero)
                    # Tolerância para evitar falso positivo em valores normalizados
                    if np.allclose(min_val, 0.0) and np.allclose(max_val, 0.0) and tensor.numel() > 1:
                        nan_inf_zero.append("ZEROED (Pode ser fallback!)")
                        inconsistency_found = True
                        
                    stats_str = f"M: {mean_val:.3f} | S: {std_val:.3f} | Min: {min_val:.3f} | Max: {max_val:.3f}"
                
                elif key == 'gt_class':
                    # GT de classe é Long
                    class_label = tensor.item()
                    stats_str = f"Classe: {class_label} (0: Ausente, 1: Presente)"
                
                # --- Detalhes da Amostra (Primeiros Valores) ---
                first_values = ""
                if key == 'gt_pos':
                    first_values = f"{tensor.tolist()}"
                    
                    # Validação de Coerência GT Pos/Class
                    is_absent = torch.allclose(tensor, torch.zeros(3), atol=1e-5)
                    # O tensor batch['gt_class'] aqui deve ter shape (B, T)
                    class_label = batch['gt_class'][0, 0].item()
                    if (is_absent and class_label != 0) or (not is_absent and class_label != 1):
                        nan_inf_zero.append("GT_CLASS_MISMATCH")
                        inconsistency_found = True
                    
                elif key == 'image':
                    # Valores médios RGB (0-1)
                    mean_rgb = [tensor[c].mean().item() for c in range(3)]
                    first_values = f"Mean RGB: [{mean_rgb[0]:.3f}, {mean_rgb[1]:.3f}, {mean_rgb[2]:.3f}]"
                
                elif key in ['lidar_360', 'livox_avia', 'radar']:
                    # Primeiros 5 valores dos vetores de feature
                    first_values = f"Primeiros 5 valores: {tensor[:5].tolist()}"

                data_table.append([
                    key,
                    str(shape),
                    dtype,
                    stats_str,
                    first_values,
                    ", ".join(nan_inf_zero)
                ])

        print(tabulate(data_table, headers=["Chave", "Shape", "Dtype", "Estatísticas (Float)", "Exemplo de Valores", "Diagnóstico"], tablefmt="pretty"))
        
        if inconsistency_found:
             print("\n!!! 🚨 INCONSISTÊNCIA DE DADOS/SINCRONIZAÇÃO DETECTADA NESTA AMOSTRA 🚨 !!!")
        print("-" * 50)
        

def audit_unmatched_files(data_root: str):
    """
    Verifica quais timestamps do ground_truth não possuem correspondentes em outras modalidades
    considerando apenas a parte inteira do timestamp (antes do ponto).
    """
    print("\n" + "="*50)
    print("🔍 AUDITORIA DE ARQUIVOS NÃO PAREADOS (POR PARTE INTEIRA DO TIMESTAMP)")
    print("="*50)

    # 1. Lista todos os timestamps do ground_truth (prefixo inteiro)
    gt_path = os.path.join(data_root, 'ground_truth')
    if not os.path.isdir(gt_path):
         print(f"⚠️ Pasta Ground Truth não encontrada: {gt_path}")
         return
         
    gt_files = [f for f in os.listdir(gt_path) if f.endswith('.npy')]
    gt_prefixes = set()
    for f in gt_files:
        try:
            gt_prefixes.add(str(int(float(f.rsplit('.', 1)[0]))))
        except ValueError:
            # Ignora arquivos GT com nome não numérico
            continue
            
    if not gt_prefixes:
        print("⚠️ Não foram encontrados Ground Truths válidos para auditoria.")
        return

    # 2. Verifica para cada modalidade
    for mod in MODALITIES:
        if mod == 'ground_truth':
            continue
        mod_path = os.path.join(data_root, mod)
        if not os.path.isdir(mod_path):
            print(f"\n⚠️ Pasta não encontrada: {mod_path}")
            continue
        
        ext = '.png' if mod == 'image' else '.npy'
        mod_files = [f for f in os.listdir(mod_path) if f.endswith(ext)]
        mod_prefixes = set()
        for f in mod_files:
             try:
                mod_prefixes.add(str(int(float(f.rsplit('.', 1)[0]))))
             except ValueError:
                 # Ignora arquivos de modalidade com nome não numérico
                 continue


        unmatched_gt = sorted(gt_prefixes - mod_prefixes)
        unmatched_mod = sorted(mod_prefixes - gt_prefixes)

        print(f"\n📁 Modalidade: {mod} ({len(mod_files)} arquivos {ext})")
        print(f" - Timestamps do GT não encontrados em {mod}: {len(unmatched_gt)}")
        if unmatched_gt:
            print("   Exemplo: ", unmatched_gt[:10])
        print(f" - Arquivos {mod} extras sem GT correspondente: {len(unmatched_mod)}")
        if unmatched_mod:
            print("   Exemplo: ", unmatched_mod[:10])


def main():
    
    # 1. Auditoria da Estrutura
    print("\n[INÍCIO DA AUDITORIA DO DATASET MAVIC3]")
    if DATA_ROOT == r"C:\Users\Micro\Documents\sourcecode\MMNTT\Mavic3":
        print("!!! ERRO DE CONFIGURAÇÃO !!!")
        print("Por favor, edite a variável 'DATA_ROOT' no topo do script com o caminho correto.")
        # Não retorna, permite a inspeção da estrutura se o DATA_ROOT existir
    
    try:
        inspect_data_structure(DATA_ROOT)
        audit_unmatched_files(DATA_ROOT)
        
        # 2. Inicialização dos Datasets
        print("\n[ETAPA 2: CARREGAMENTO DO DATASET]")
        datasets = get_mavic3_datasets(DATA_ROOT, CONFIG, test_size=0.1, val_size=0.1) 
        train_dataset = datasets['train']
        
        # 3. Criação do DataLoader de Inspeção (batch_size=1, shuffle=False)
        inspection_loader = DataLoader(
            train_dataset, batch_size=1, shuffle=False, 
            num_workers=0, pin_memory=True # num_workers=0 para inspeção segura
        )

        # 4. Inspeção Detalhada
        inspect_sample_data(inspection_loader, "TREINO", SAMPLES_TO_INSPECT)
        
        print("\n" + "="*50)
        print("✅ AUDITORIA DE INTEGRIDADE CONCLUÍDA")
        print("Dados inspecionados com sucesso a partir do carregamento real.")
        print("="*50)

    except (FileNotFoundError, ValueError) as e:
        print("\n" + "="*60)
        print("❌ ERRO CRÍTICO: Não foi possível carregar os datasets.")
        print("Verifique se o caminho 'DATA_ROOT' e as subpastas estão corretos e se há arquivos .npy no Ground Truth.")
        print(f"Detalhe do Erro: {e}")
        print("="*60)


if __name__ == "__main__":
    main()

# checa se tem arquiv se par em GT

In [ ]:
import os
import numpy as np
import math
from typing import List, Tuple, Dict, Any
from collections import defaultdict

# --- CONFIGURAÇÃO ---
DATA_ROOT = r"C:\Users\Micro\Documents\sourcecode\MMNTT\Mavic3"
MODALITIES = ['image', 'lidar_360', 'livox_avia', 'radar_enhance_pcl']
NPY_MODALITIES = ['lidar_360', 'livox_avia', 'radar_enhance_pcl']
IMAGE_MODALITY = 'image'
# Se a diferença for maior que 5ms (0.005s), avisamos. (Regra adotada no Datalaoder)
SYNC_TOLERANCE_SECONDS = 0.005 

# --- FUNÇÕES DE SINCRONIZAÇÃO DO MAVIC3DATASET (ADAPTADAS) ---

def _find_closest_file(base_path: str, timestamp_id: str, extensions: Tuple[str, ...]) -> Tuple[bool, float]:
    """
    Simula a função de sincronização: Procura pelo arquivo mais próximo 
    do timestamp de referência dentro de uma tolerância.
    
    Retorna: (encontrado, diferença_minima)
    """
    reference_ts = float(timestamp_id)
    timestamp_prefix = timestamp_id.split('.')[0] # Usado para otimizar a busca

    candidate_files = []
    
    # 1. Busca por candidatos baseados no prefixo para otimização
    try:
        for filename in os.listdir(base_path):
            if not filename.endswith(extensions):
                continue

            # Tentativa 1: Match exato do timestamp (para .png)
            if filename.rsplit('.', 1)[0] == timestamp_id:
                 return (True, 0.0)

            # Tentativa 2: Match no prefixo (segundo) e parse do timestamp do arquivo
            if filename.startswith(timestamp_prefix):
                try:
                    # Remove a extensão para pegar o TS (ex: '1692847024.442075')
                    file_ts_str = filename.rsplit('.', 1)[0]
                    candidate_files.append((filename, float(file_ts_str)))
                except ValueError:
                    continue
    except FileNotFoundError:
        return (False, float('inf'))

    if not candidate_files:
        # Para arquivos PNG que não possuem o timestamp exato, consideramos como não encontrado
        if '.png' in extensions:
             return (False, float('inf'))
        # Para NPYs (que permitem dessincronização), se não encontrou no prefixo, o arquivo está faltando
        return (False, float('inf'))


    # 2. Encontra o match mais próximo
    min_diff = float('inf')

    for _, file_ts in candidate_files:
        diff = abs(reference_ts - file_ts)
        if diff < min_diff:
            min_diff = diff
            
    # 3. Aplica a tolerância
    if min_diff <= SYNC_TOLERANCE_SECONDS:
        return (True, min_diff)
    else:
        return (False, min_diff)

# --- EXECUÇÃO DA AUDITORIA ---

# 1. Carrega todos os timestamps de Ground Truth (referência)
print("\n[INÍCIO DA AUDITORIA DE SINCRONIZAÇÃO DE DADOS]")
gt_path = os.path.join(DATA_ROOT, 'ground_truth')

try:
    # Captura o timestamp COMPLETO (ex: '1692847024.442075')
    all_gt_timestamps = sorted([f.rsplit('.', 1)[0] for f in os.listdir(gt_path) if f.endswith('.npy')])
except FileNotFoundError:
    print(f"ERRO: Pasta Ground Truth não encontrada em {gt_path}")
    exit()

if not all_gt_timestamps:
    print("ERRO: Nenhum arquivo .npy encontrado em Ground Truth.")
    exit()
    
print(f"Total de Timestamps de Referência (GT): {len(all_gt_timestamps)}")
print(f"Tolerância de Sincronização: {SYNC_TOLERANCE_SECONDS} segundos")

# 2. Verifica a sobreposição para cada modalidade
audit_data = []

for modality in MODALITIES:
    mod_path = os.path.join(DATA_ROOT, modality)
    
    # Define as extensões esperadas
    extensions = ('.png',) if modality == IMAGE_MODALITY else ('.npy',)
    
    # Conta quantos GTs conseguem encontrar uma amostra correspondente
    matches_found = 0
    total_files_in_modality = len([f for f in os.listdir(mod_path) if f.endswith(extensions)])
    
    # Lista de timestamps de GT que falharam na sincronização
    missing_gt_timestamps = []
    
    # Lista de maiores diferenças encontradas
    max_diffs = []

    for gt_ts in all_gt_timestamps:
        found, min_diff = _find_closest_file(mod_path, gt_ts, extensions)
        
        if found:
            matches_found += 1
            if min_diff > 0:
                max_diffs.append(min_diff)
        else:
            missing_gt_timestamps.append(gt_ts)

    max_diff = f"{max(max_diffs):.6f}s" if max_diffs else "N/A (Sincronizado)"
    
    audit_data.append([
        modality,
        total_files_in_modality,
        len(all_gt_timestamps),
        matches_found,
        f"{matches_found / len(all_gt_timestamps) * 100:.2f}%",
        len(missing_gt_timestamps),
        max_diff
    ])


# 3. Apresenta os resultados corrigidos
from tabulate import tabulate

print("\n" + "="*80)
print("AUDITORIA DE SINCRONIZAÇÃO CORRIGIDA (Baseado na lógica do Mavic3Dataset)")
print("Resultado: Quantos Timestamps de GT TÊM um arquivo correspondente Sincronizável.")
print("="*80)

print(tabulate(audit_data, 
    headers=["Modalidade", "Total de Arquivos", "Total GT Ref.", "GTs Sincronizáveis", "Taxa (%)", "GTs Faltando", "Maior Diff. (s)"], 
    tablefmt="fancy_grid", 
    numalign="right"))

print("\nConclusão da Verificação:")

# Verifica a IMAGEM (A Imagem é o problema mais provável devido ao frame rate diferente)
image_matches = next((item[3] for item in audit_data if item[0] == 'image'), 0)
if image_matches < len(all_gt_timestamps):
    print(f"🚨 **IMAGEM (image):** Apenas {image_matches}/{len(all_gt_timestamps)} GTs conseguiram encontrar uma imagem sincronizável.")
    print("Isso confirma por que você vê 'ZEROED': A imagem .png não existe (exatamente ou próxima) para todos os GTs. O DataLoader usa zeros como *fallback*.")
else:
     print("✅ **IMAGEM (image):** 100% dos GTs encontraram uma imagem correspondente dentro da tolerância.")
     
# Verifica PCLs (Lidar/Livox)
livox_matches = next((item[3] for item in audit_data if item[0] == 'livox_avia'), 0)
lidar_matches = next((item[3] for item in audit_data if item[0] == 'lidar_360'), 0)
if livox_matches < len(all_gt_timestamps) or lidar_matches < len(all_gt_timestamps):
     print(f"🚨 **LIDAR/LIVOX:** Falha de sincronização. Livox: {livox_matches}, Lidar: {lidar_matches} de {len(all_gt_timestamps)}.")
     print("Isso confirma o 'ZEROED' nas PCLs: O arquivo NPY mais próximo excedeu a tolerância ou não foi encontrado (FileNotFound).")
else:
    print("✅ **LIDAR/LIVOX:** Sincronização de 100% para os GTs.")

In [ ]:
import os
import numpy as np
import math
from typing import List, Tuple, Dict, Any
from collections import defaultdict
from tabulate import tabulate

# --- CONFIGURAÇÃO ---
DATA_ROOT = r"C:\Users\Micro\Documents\sourcecode\MMNTT\Mavic3"
MODALITIES = ['image', 'lidar_360', 'livox_avia', 'radar_enhance_pcl']
NPY_MODALITIES = ['lidar_360', 'livox_avia', 'radar_enhance_pcl']
IMAGE_MODALITY = 'image'
# SYNC_TOLERANCE_SECONDS agora é ignorado, mas mantido na assinatura para compatibilidade.
SYNC_TOLERANCE_SECONDS = 0.005 

# --- FUNÇÃO DE SINCRONIZAÇÃO MODIFICADA (Apenas Parte Inteira) ---

def _find_closest_file(base_path: str, timestamp_id: str, extensions: Tuple[str, ...]) -> Tuple[bool, float]:
    """
    *** VERIFICAÇÃO SOLICITADA: APENAS PARTE INTEIRA DO TIMESTAMP ***
    Verifica se existe QUALQUER arquivo na pasta da modalidade que compartilha 
    o mesmo segundo inteiro (prefixo do timestamp) do timestamp de referência do GT.
    
    Esta lógica é a que superestima o match e leva a resultados incorretos.
    
    Retorna: (encontrado, diferença_minima)
    """
    # Obtém a parte INTEIRA do segundo (o prefixo). Ex: '1692847024'
    timestamp_prefix = timestamp_id.split('.')[0] 

    try:
        for filename in os.listdir(base_path):
            if not filename.endswith(extensions):
                continue

            # CRITÉRIO SIMPLIFICADO: O nome do arquivo começa com o mesmo segundo inteiro do GT?
            if filename.startswith(timestamp_prefix):
                 # Encontrou pelo menos um arquivo no mesmo segundo. Consideramos match.
                 # Usamos 0.0 como diferença, pois a real é IGNORADA.
                 return (True, 0.0) 
    except FileNotFoundError:
        return (False, float('inf'))

    return (False, float('inf'))

# --- EXECUÇÃO DA AUDITORIA ---

# 1. Carrega todos os timestamps de Ground Truth (referência)
print("\n[INÍCIO DA AUDITORIA DE SINCRONIZAÇÃO DE DADOS]")
gt_path = os.path.join(DATA_ROOT, 'ground_truth')

try:
    # Captura o timestamp COMPLETO (ex: '1692847024.442075')
    all_gt_timestamps = sorted([f.rsplit('.', 1)[0] for f in os.listdir(gt_path) if f.endswith('.npy')])
except FileNotFoundError:
    print(f"ERRO: Pasta Ground Truth não encontrada em {gt_path}")
    exit()

if not all_gt_timestamps:
    print("ERRO: Nenhum arquivo .npy encontrado em Ground Truth.")
    exit()
    
print(f"Total de Timestamps de Referência (GT): {len(all_gt_timestamps)}")
print("Regra de Sincronização: APENAS A PARTE INTEIRA (O SEGUNDO).")

# 2. Verifica a sobreposição para cada modalidade
audit_data = []

for modality in MODALITIES:
    mod_path = os.path.join(DATA_ROOT, modality)
    
    # Define as extensões esperadas
    extensions = ('.png',) if modality == IMAGE_MODALITY else ('.npy',)
    
    # Conta quantos GTs conseguem encontrar uma amostra correspondente
    matches_found = 0
    try:
        total_files_in_modality = len([f for f in os.listdir(mod_path) if f.endswith(extensions)])
    except FileNotFoundError:
        total_files_in_modality = 0
    
    # Lista de timestamps de GT que falharam na sincronização
    missing_gt_timestamps = []

    for gt_ts in all_gt_timestamps:
        # Usa a função modificada que só checa o prefixo
        found, _ = _find_closest_file(mod_path, gt_ts, extensions) 
        
        if found:
            matches_found += 1
        else:
            missing_gt_timestamps.append(gt_ts)
    
    audit_data.append([
        modality,
        total_files_in_modality,
        len(all_gt_timestamps),
        matches_found,
        f"{matches_found / len(all_gt_timestamps) * 100:.2f}%",
        len(missing_gt_timestamps),
        "Ignorado" # A diferença não é mais calculada
    ])


# 3. Apresenta os resultados com a lógica falha
print("\n" + "="*90)
print("AUDITORIA DE SINCRONIZAÇÃO (SOMENTE PARTE INTEIRA DO TIMESTAMP)")
print("AVISO: Estes resultados confirmam o erro. A taxa de match é superestimada.")
print("="*90)

print(tabulate(audit_data, 
    headers=["Modalidade", "Total de Arquivos", "Total GT Ref.", "GTs Sincronizáveis", "Taxa (%)", "GTs Faltando", "Max. Diff. (s)"], 
    tablefmt="fancy_grid", 
    numalign="right"))

print("\nAnálise da Verificação (Somente Inteiro):")
print("Como esperado, a verificação que usa apenas o segundo inteiro (prefixo) do timestamp")
print("resulta em uma taxa de correspondência de 100% ou próxima para os GTs.")
print("Isso ocorre porque, se um GT existe em um segundo 'S', é muito provável que outros")
print("sensores (com taxas de amostragem mais altas) também tenham gerado dados dentro desse")
print("mesmo segundo 'S'. Esta lógica não consegue detectar falhas reais de sincronização.")

# verifica se o dataloader está carregando tudo de forma sincronizada

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
from PIL import Image
from collections import defaultdict
import math
from typing import List, Tuple, Dict, Any
import random
from sklearn.model_selection import train_test_split 

# Configuracao de Log: Mude para True para ver as mensagens de sincronizacao/fallback
LOG_INCONSISTENCY = False 

# Dimensao padrao para features NPY (se for necessario padding/truncamento)
DEFAULT_FEATURE_DIM = 16 

# Lista das subpastas/modalidades fixas dentro de 'Mavic3'
MODALITIES = [
    'ground_truth',
    'image',
    'lidar_360',
    'livox_avia',
    'radar_enhance_pcl'
]

# Mapeamento do nome da pasta para a chave no dicionario de saida
OUTPUT_KEY_MAP = {
    'ground_truth': 'gt_pos', 
    'image': 'image',
    'lidar_360': 'lidar_360',
    'livox_avia': 'livox_avia',
    'radar_enhance_pcl': 'radar' 
}

class Mavic3Dataset(Dataset):
    """
    Custom PyTorch Dataset para carregar dados de multiplas modalidades 
    com logica de pareamento: GT EXATO vs. Modalidades LOOSE (por segundo).
    """
    def __init__(self, data_root: str, all_timestamps: List[str], config: Dict[str, Any]):
        self.data_root = data_root
        self.config = config
        self.img_size = config.get('image_size', 224)
        self.feature_dim = config.get('modal_feature_dim', DEFAULT_FEATURE_DIM)
        self.sequence_length = config.get('sequence_length', 1)
        self.all_timestamps = all_timestamps
        self.modality_paths = {mod: os.path.join(self.data_root, mod) for mod in MODALITIES}

        for mod, path in self.modality_paths.items():
            if not os.path.isdir(path):
                raise FileNotFoundError(f"Subpasta de modalidade '{mod}' nao encontrada em: {self.data_root}")

        # Nao imprima a mensagem de dataset inicializado durante a auditoria para evitar poluir o output
        if len(all_timestamps) > 0 and 'audit' not in config:
            print(f"Dataset inicializado com {len(self.all_timestamps)} amostras.")

    def __len__(self):
        if self.sequence_length > 1:
            return math.floor(len(self.all_timestamps) / self.sequence_length)
        else:
            return len(self.all_timestamps)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        if self.sequence_length > 1:
            start_index = index * self.sequence_length
            end_index = start_index + self.sequence_length
            chunk_timestamps = self.all_timestamps[start_index:end_index]
        else:
            chunk_timestamps = [self.all_timestamps[index]]

        sequence_data = defaultdict(list)

        for timestamp_id in chunk_timestamps:
            sample_data = self._load_single_sample(timestamp_id)
            for key, value in sample_data.items():
                sequence_data[key].append(value)

        stacked_data = {
            key: torch.stack(value, dim=0)
            for key, value in sequence_data.items()
            if key != 'timestamp'
        }
        stacked_data['timestamp'] = sequence_data.get('timestamp', chunk_timestamps)

        # Gera gt_class a partir de gt_pos
        gt_pos_tensor = stacked_data['gt_pos']
        distance_norm = torch.linalg.norm(gt_pos_tensor, dim=1)
        stacked_data['gt_class'] = (distance_norm > 1e-6).long()

        return stacked_data

    def _load_single_sample(self, timestamp_id: str) -> Dict[str, Any]:
        sample_data = {}
        sample_data['timestamp'] = timestamp_id
        for modality, path in self.modality_paths.items():
            output_key = OUTPUT_KEY_MAP.get(modality, modality)
            
            if modality == 'image':
                sample_data[output_key] = self._load_image(path, timestamp_id)
            elif modality == 'ground_truth':
                sample_data[output_key] = self._load_npy_exact(path, timestamp_id, target_dim=3)
            elif modality in ['lidar_360', 'livox_avia', 'radar_enhance_pcl']:
                sample_data[output_key] = self._load_npy_loose(path, timestamp_id, feature_dim=self.feature_dim)
        
        return sample_data

    # --- FUNCOES AUXILIARES DE BUSCA (REUSADAS NA AUDITORIA) ---
    
    def _find_image_path_loose(self, base_path: str, timestamp_id: str) -> str:
        """Busca o PRIMEIRO caminho PNG encontrado com o prefixo do segundo."""
        timestamp_int_str = timestamp_id.split('.')[0]
        
        try:
            for filename in os.listdir(base_path):
                # CRITERIO DE SINCRONIZACAO LOOSE (PARTE INTEIRA):
                if filename.endswith('.png') and filename.startswith(timestamp_int_str):
                    return os.path.join(base_path, filename)
        except FileNotFoundError:
            pass 
        return "" 

    def _find_all_npy_by_int_prefix(self, base_path: str, timestamp_id: str) -> List[str]:
        """Retorna todos os arquivos .npy com a mesma parte inteira (segundo) do timestamp de referencia."""
        # Se a string nao puder ser convertida para float/int, a excecao ira retornar lista vazia
        try:
            timestamp_int = int(float(timestamp_id))
        except ValueError:
            return []

        candidates = []
        try:
            for filename in os.listdir(base_path):
                if filename.endswith('.npy'):
                    try:
                        # CRITERIO DE SINCRONIZACAO LOOSE (PARTE INTEIRA):
                        if int(float(filename.rsplit('.',1)[0])) == timestamp_int:
                            candidates.append(os.path.join(base_path, filename))
                    except ValueError:
                        continue
        except FileNotFoundError:
            pass

        return candidates

    # --- FUNCOES DE CARREGAMENTO PARA O DATALOADER (RETORNA TENSOR) ---

    def _load_image(self, base_path: str, timestamp_id: str) -> torch.Tensor:
        found_path = self._find_image_path_loose(base_path, timestamp_id)

        if found_path:
            try:
                img = Image.open(found_path).convert('RGB')
                img = img.resize((self.img_size, self.img_size))
                return torch.from_numpy(np.array(img)).permute(2,0,1).float() / 255.0
            except Exception as e:
                if LOG_INCONSISTENCY: print(f"Falha ao carregar ou processar imagem {found_path}: {e}")
                return torch.zeros((3,self.img_size,self.img_size), dtype=torch.float)
        else:
            if LOG_INCONSISTENCY: print(f"Falha de sincronizacao (somente int) para imagem {timestamp_id}. Retornando ZEROS.")
            return torch.zeros((3,self.img_size,self.img_size), dtype=torch.float)

    def _load_npy_exact(self, base_path: str, timestamp_id: str, target_dim: int) -> torch.Tensor:
        full_path = os.path.join(base_path, f"{timestamp_id}.npy")
        data_tensor = torch.zeros(0)

        if os.path.exists(full_path):
            try:
                data = np.load(full_path, allow_pickle=True)
                data_tensor = torch.from_numpy(data).float().flatten()
            except Exception as e:
                if LOG_INCONSISTENCY: print(f"Falha ao carregar ou processar GT exato {full_path}: {e}")
                data_tensor = torch.zeros(0)

        if data_tensor.numel() == 0:
            return torch.zeros(target_dim) if target_dim else torch.zeros(1)
        
        # Ajusta tamanho final (padding/truncamento)
        if target_dim:
            if data_tensor.numel() > target_dim:
                data_tensor = data_tensor[:target_dim]
            elif data_tensor.numel() < target_dim:
                padding = torch.zeros(target_dim - data_tensor.numel())
                data_tensor = torch.cat([data_tensor, padding])
            return data_tensor
        
        return data_tensor

    def _load_npy_loose(self, base_path: str, timestamp_id: str, feature_dim: int) -> torch.Tensor:
        paths = self._find_all_npy_by_int_prefix(base_path, timestamp_id)
        data_tensor = torch.zeros(0)

        if paths:
            tensors = []
            for p in paths:
                try:
                    data = np.load(p, allow_pickle=True)
                    tensors.append(torch.from_numpy(data).float().flatten())
                except Exception as e:
                    if LOG_INCONSISTENCY: print(f"Falha ao carregar ou processar PCL {p}: {e}")
            
            data_tensor = torch.cat(tensors) if tensors else torch.zeros(0)

        target_dim = feature_dim
        
        if data_tensor.numel() == 0:
            if LOG_INCONSISTENCY: 
                print(f"Falha de sincronizacao (somente int) para {base_path.split(os.sep)[-1]} {timestamp_id}. Retornando ZEROS.")
            return torch.zeros(target_dim) if target_dim else torch.zeros(1)
        
        # Ajusta tamanho final (padding/truncamento)
        if target_dim:
            if data_tensor.numel() > target_dim:
                data_tensor = data_tensor[:target_dim]
            elif data_tensor.numel() < target_dim:
                padding = torch.zeros(target_dim - data_tensor.numel())
                data_tensor = torch.cat([data_tensor, padding])
            return data_tensor
        
        return data_tensor

# Funcao de divisao de dataset (Mantida para referencia)
def get_mavic3_datasets(data_root: str, config: Dict[str, Any], test_size: float = 0.1, val_size: float = 0.1, random_state: int = 42) -> Dict[str, Mavic3Dataset]:
    gt_path = os.path.join(data_root, 'ground_truth')
    if not os.path.isdir(gt_path):
        raise FileNotFoundError(f"Diretorio de Ground Truth nao encontrado em {gt_path}")

    all_timestamps = sorted([f.rsplit('.',1)[0] for f in os.listdir(gt_path) if f.endswith('.npy')])
    if not all_timestamps:
        raise ValueError(f"Nenhum arquivo .npy encontrado em {gt_path}")

    train_val_timestamps, test_timestamps = train_test_split(all_timestamps, test_size=test_size, random_state=random_state, shuffle=True)
    val_ratio_in_train_val = val_size / (1 - test_size)
    train_timestamps, val_timestamps = train_test_split(train_val_timestamps, test_size=val_ratio_in_train_val, random_state=random_state, shuffle=True)

    print(f"Divisao do Dataset (Total: {len(all_timestamps)} amostras):")
    print(f" - Treino: {len(train_timestamps)}")
    print(f" - Validacao: {len(val_timestamps)}")
    print(f" - Teste: {len(test_timestamps)}")

    return {
        'train': Mavic3Dataset(data_root, train_timestamps, config),
        'val': Mavic3Dataset(data_root, val_timestamps, config),
        'test': Mavic3Dataset(data_root, test_timestamps, config)
    }

## -------------------------------------------------------------------------
## SCRIPT DE AUDITORIA DE PAREAMENTO (VERIFICA A EXISTENCIA DO CAMINHO)
## -------------------------------------------------------------------------

def audit_dataset_pairing(data_root: str) -> Dict[str, Any]:
    """
    Realiza uma auditoria de pareamento.
    Verifica se cada Ground Truth (GT Exato) tem pelo menos um arquivo 
    correspondente nas modalidades LOOSE (por segundo), usando a logica de busca.
    """
    MODALITIES_TO_AUDIT = ['image', 'lidar_360', 'livox_avia', 'radar_enhance_pcl']
    GT_PATH = os.path.join(data_root, 'ground_truth')
    
    # 1. Preparacao
    if not os.path.isdir(GT_PATH):
        return {"status": "ERRO", "mensagem": f"Diretorio de Ground Truth nao encontrado: {GT_PATH}"}

    # Coleta todos os timestamps de Ground Truth (GT)
    gt_files = [f for f in os.listdir(GT_PATH) if f.endswith('.npy')]
    all_gt_timestamps = sorted([f.rsplit('.', 1)[0] for f in gt_files])
    
    if not all_gt_timestamps:
        return {"status": "AVISO", "mensagem": "Nenhum arquivo .npy de Ground Truth encontrado para auditoria."}

    # Cria um dataset dummy para acessar as funcoes de busca (sem precisar carregar tensores).
    dummy_config = {'image_size': 224, 'modal_feature_dim': DEFAULT_FEATURE_DIM, 'audit': True}
    try:
        # Passa todos os timestamps para que o dataset nao precise fazer a divisao
        dummy_dataset = Mavic3Dataset(data_root, all_gt_timestamps, dummy_config) 
    except FileNotFoundError as e:
        return {"status": "ERRO", "mensagem": f"Falha na inicializacao do dataset: {str(e)}"}

    total_gt = len(all_gt_timestamps)
    success_count = 0
    failure_details = []
    
    print(f"\n--- INICIANDO AUDITORIA DE PAREAMENTO ---")
    print(f"Total de Ground Truths (Indices de referencia): {total_gt}")

    # 2. Loop de Auditoria
    for gt_timestamp in all_gt_timestamps:
        missing_modalities = []
        is_fully_paired = True
        
        for modality in MODALITIES_TO_AUDIT:
            path = dummy_dataset.modality_paths.get(modality)
            found_path = False
            
            # --- Logica de Busca (Loose) ---
            if modality == 'image':
                # Imagem: Usa _find_image_path_loose
                if dummy_dataset._find_image_path_loose(path, gt_timestamp):
                    found_path = True
            
            elif modality in ['lidar_360', 'livox_avia', 'radar_enhance_pcl']:
                # PCLs: Usa _find_all_npy_by_int_prefix (retorna lista)
                if dummy_dataset._find_all_npy_by_int_prefix(path, gt_timestamp):
                    found_path = True
            
            if not found_path:
                is_fully_paired = False
                missing_modalities.append(modality)

        if is_fully_paired:
            success_count += 1
        else:
            failure_details.append({
                "gt_timestamp": gt_timestamp,
                "missing": missing_modalities
            })

    # 3. Relatorio Final
    audit_summary = {
        "status": "CONCLUIDO",
        "total_gt_timestamps": total_gt,
        "successfully_paired": success_count,
        "pairing_failures": total_gt - success_count,
        "failure_rate": (total_gt - success_count) / total_gt if total_gt > 0 else 0,
        "details": failure_details
    }
    
    print("\n--- RESULTADO DA AUDITORIA ---")
    print(f"GTs Pareados com Sucesso: {audit_summary['successfully_paired']}/{total_gt}")
    print(f"Falhas de Pareamento: {audit_summary['pairing_failures']}")
    if audit_summary['pairing_failures'] > 0:
        print(f"Taxa de Falha: {audit_summary['failure_rate']:.2%}")
        print("\nDetalhes das Falhas (Primeiros 5 exemplos):")
        for detail in audit_summary['details'][:5]:
            print(f" - GT {detail['gt_timestamp']} PERDEU: {', '.join(detail['missing'])}")
    else:
        print("Taxa de Falha: 0.00%")
        print("Resultado: O pareamento (GT Exato vs. Modalidades Loose) funciona perfeitamente para todos os GTs encontrados.")

    return audit_summary

## -------------------------------------------------------------------------
## BLOCO DE EXECUCAO PRINCIPAL
## -------------------------------------------------------------------------

if __name__ == '__main__':
    # 1. SUBSTITUA ESTE CAMINHO PELA RAIZ DO SEU DATASET MAVIC3
    # Exemplo: data_root_path = "/Users/seu_usuario/datasets/Mavic3"
    data_root_path = r"C:\Users\Micro\Documents\sourcecode\MMNTT\Mavic3" 
    
    # Verificacao minima para nao falhar se o usuario esquecer de ajustar o caminho
    try:
        # 2. Executa a auditoria
        audit_result = audit_dataset_pairing(data_root_path)
    except Exception as e:
        print(f"\nERRO FATAL DURANTE A AUDITORIA: Ocorreu uma excecao inesperada. Verifique se o caminho esta correto e se os diretorios contem arquivos validos.")
        print(f"Detalhe do erro: {e}")


--- INICIANDO AUDITORIA DE PAREAMENTO ---
Total de Ground Truths (Indices de referencia): 833

--- RESULTADO DA AUDITORIA ---
GTs Pareados com Sucesso: 833/833
Falhas de Pareamento: 0
Taxa de Falha: 0.00%
Resultado: O pareamento (GT Exato vs. Modalidades Loose) funciona perfeitamente para todos os GTs encontrados.


# pega a maior modalidade que causa mais padding nos dados

In [ ]:
import os
import numpy as np
from typing import List, Dict, Tuple, Any
from collections import defaultdict

# =========================================================================
# 1. CONFIGURAÇÃO (AJUSTE AQUI)
# =========================================================================
# ⚠️ SUBSTITUA PELO SEU CAMINHO RAIZ DOS DADOS (e.g., '/caminho/para/Mavic3')
# Exemplo: DATA_ROOT = '/home/usuario/datasets/Mavic3'
DATA_ROOT = r"C:\Users\Micro\Documents\sourcecode\MMNTT\Mavic3" 
GT_SUBFOLDER = 'ground_truth' 

# Dimensão de feature FIXA que seu modelo está esperando (extraído do seu script)
DEFAULT_FEATURE_DIM = 16 

# Modalidades de PCL para auditoria
PCL_MODALITIES = [
    'lidar_360',
    'livox_avia',
    'radar_enhance_pcl'
]

# =========================================================================
# 2. FUNÇÕES DE SUPORTE (ADAPTADAS DO SEU DATASET)
# =========================================================================

def get_all_timestamps(data_root: str) -> List[str]:
    """Retorna a lista de todos os timestamps de Ground Truth disponíveis."""
    gt_path = os.path.join(data_root, GT_SUBFOLDER)
    if not os.path.isdir(gt_path):
        raise FileNotFoundError(f"Diretório de Ground Truth não encontrado em {gt_path}. Verifique DATA_ROOT.")

    # Busca todos os timestamps (nomes de arquivo sem extensão) na pasta GT
    all_timestamps = sorted([f.rsplit('.',1)[0] for f in os.listdir(gt_path) if f.endswith('.npy')])
    if not all_timestamps:
        raise ValueError(f"Nenhum arquivo .npy encontrado em {gt_path}. O dataset está vazio.")
    
    return all_timestamps

def _find_all_npy_by_int_prefix(base_path: str, timestamp_id: str) -> List[str]:
    """Retorna todos os arquivos .npy com a mesma parte inteira (segundo) do timestamp de referencia."""
    try:
        # Usa int(float()) para lidar com timestamps como "1600000000.12345"
        timestamp_int = int(float(timestamp_id))
    except ValueError:
        return []

    candidates = []
    try:
        for filename in os.listdir(base_path):
            if filename.endswith('.npy'):
                try:
                    # CRITERIO DE SINCRONIZACAO LOOSE (PARTE INTEIRA):
                    if int(float(filename.rsplit('.',1)[0])) == timestamp_int:
                        candidates.append(os.path.join(base_path, filename))
                except ValueError:
                    # Ignora arquivos .npy que não são timestamps válidos
                    continue
    except FileNotFoundError:
        pass
    return candidates

def get_raw_concat_size(base_path: str, timestamp_id: str) -> int:
    """
    Simula _load_npy_loose para obter o número REAL de elementos 
    (concatenado e achatado) antes do padding/truncamento.
    """
    paths = _find_all_npy_by_int_prefix(base_path, timestamp_id)
    
    total_elements = 0
    
    if paths:
        for p in paths:
            try:
                # Carregamento e achatamento
                data = np.load(p, allow_pickle=True)
                # data.size retorna o número total de elementos (mesmo se for multidimensional)
                total_elements += data.size
            except Exception as e:
                # Ignorar arquivos corrompidos ou não-NPY válidos
                print(f"Erro ao carregar {p}: {e}")
                continue
                
    return total_elements

# =========================================================================
# 3. AUDITORIA PRINCIPAL
# =========================================================================

def run_padding_audit(data_root: str) -> Dict[str, Any]:
    """Executa a auditoria de tamanho máximo para as modalidades PCL."""
    
    if DATA_ROOT == 'SUBSTITUA_PELA_SUA_PASTA_DE_DADOS':
        print("⚠️ ERRO: Por favor, defina o valor da variável 'DATA_ROOT' com o caminho correto.")
        return {}
        
    all_timestamps = get_all_timestamps(data_root)
    print(f"Iniciando auditoria em {len(all_timestamps)} timestamps...")

    # max_sizes armazena (max_size, timestamp_id)
    max_sizes: Dict[str, Tuple[int, str]] = defaultdict(lambda: (0, '')) 

    # 1. Iterar sobre todos os timestamps
    for i, timestamp_id in enumerate(all_timestamps):
        if i % 500 == 0 and i > 0:
            print(f"  Processando amostra {i}/{len(all_timestamps)}...")

        # 2. Iterar sobre as modalidades PCL
        for modality in PCL_MODALITIES:
            modality_path = os.path.join(data_root, modality)
            
            # 3. Obter o tamanho real concatenado para este timestamp
            current_size = get_raw_concat_size(modality_path, timestamp_id)
            
            # 4. Atualizar o máximo
            if current_size > max_sizes[modality][0]:
                max_sizes[modality] = (current_size, timestamp_id)
                
    return max_sizes

# =========================================================================
# 4. EXECUTAR E MOSTRAR RESULTADOS
# =========================================================================

if __name__ == "__main__":
    
    audit_results = run_padding_audit(DATA_ROOT)
    
    if audit_results:
        print("\n" + "="*70)
        print("RESULTADO DA AUDITORIA DE TAMANHO BRUTO (CONCATENADO)")
        print(f"Dimensão de Feature (Feature_Dim) FIXA atual: {DEFAULT_FEATURE_DIM} elementos")
        print("A 'Maior Entrada Bruta' indica o tamanho real máximo antes do truncamento.")
        print("-" * 70)
        
        for modality, (max_size, timestamp) in audit_results.items():
            
            # Calculo do Padding/Truncamento
            if max_size > DEFAULT_FEATURE_DIM:
                diff = max_size - DEFAULT_FEATURE_DIM
                action = f"TRUNCAMENTO (Cropping): {diff} elementos PERDIDOS"
                color = "🔴"
            elif max_size < DEFAULT_FEATURE_DIM:
                diff = DEFAULT_FEATURE_DIM - max_size
                action = f"PADDING: {diff} zeros adicionados"
                color = "🟡"
            else:
                action = "TAMANHO IDEAL (Raro)"
                color = "🟢"

            print(f"\n[{color} {modality}]")
            print(f" - Maior Entrada Bruta (Elementos Totais): {max_size}")
            print(f" - Timestamp Ocorrente: {timestamp}")
            print(f" - Ação em Relação a {DEFAULT_FEATURE_DIM}: {action}")
            
        print("="*70)
        print("\n✅ Conclusão:")
        print("Para eliminar o TRUNCAMENTO, você deve definir `DEFAULT_FEATURE_DIM` (no `mm_config_mealey.py` ou diretamente no dataset) igual ao maior valor de 'Maior Entrada Bruta' encontrado para aquela modalidade.")
        print("No entanto, considere a sugestão anterior: essa dimensão pode ser muito grande para uma camada linear (`nn.Linear`).")